# Retrieval metrics usage

## 1. What retrieval metrics measure

Retrieval metrics compare the query in `EvaluationCase.input` with the ranked list in `retrieved_documents`. List order is rank. Relevance@K measures top-K precision, Hit Rate@K detects any relevant result, MRR@K rewards the first relevant result appearing early, and nDCG@K measures overall relevant-document ordering. They do not require generated `output`.

## 2. Configure a judge

In [ ]:
from idp_eval import (
    EvaluationCase,
    EvaluationFramework,
    HitRateAtKEvaluator,
    MRRAtKEvaluator,
    NDCGAtKEvaluator,
    RelevanceAtKEvaluator,
    create_azure_judge,
)
from idp_eval.judges import AzureJudgeConfig

Applications should inject config from their own settings/secrets layer. These are placeholders. `create_gateway_judge(config=...)` works equivalently; retrieval semantics are backend-independent.

In [ ]:
azure_config = AzureJudgeConfig(
    model="your-azure-deployment",
    azure_endpoint="https://your-resource.openai.azure.com",
    tenant_id="your-tenant-id",
    client_id="your-client-id",
    client_secret="your-client-secret",
    api_version="2024-12-01-preview",
    timeout=180,
    proxy_url=None,
    verify_ssl=True,
    reasoning_effort=None,
)
judge = create_azure_judge(config=azure_config)

## 3. Build ranked `retrieved_documents`

Documents may be plain strings or mappings. For mappings, `text` is the default judged field; `document_id`, retriever `score`, and metadata remain diagnostics and are never sent to the relevance judge. Configure `document_text_key="body"` on every retrieval evaluator when another text key is needed.

In [ ]:
documents = [
    {"document_id": "doc-1", "text": "Open the account settings page and select Reset password.", "score": 0.94},
    {"document_id": "doc-2", "text": "Billing statements are available under Billing history.", "score": 0.89},
    {"document_id": "doc-3", "text": "A password-reset email expires after 30 minutes.", "score": 0.84},
    {"document_id": "doc-4", "text": "Contact support if the reset email does not arrive.", "score": 0.78},
    {"document_id": "doc-5", "text": "Update notification preferences from the profile page.", "score": 0.71},
]
case = EvaluationCase(
    case_id="retrieval-001",
    input="How do I reset my password?",
    retrieved_documents=documents,
)

## 4. Configure all four metrics

In [ ]:
framework = EvaluationFramework(
    judge=judge,
    evaluators=[
        RelevanceAtKEvaluator(k=5),
        HitRateAtKEvaluator(k=5),
        MRRAtKEvaluator(k=5),
        NDCGAtKEvaluator(k=5),
    ],
)
framework.metrics

## 5. Run evaluation

In [ ]:
results = framework.evaluate(case)
{name: {"score": result.score, "label": result.label} for name, result in results.items()}

## 6. Relevance@K interpretation

Relevance@K is the fraction of relevant documents in the evaluated top K—equivalent to Precision@K under binary relevance. Python computes `relevant_count / effective_k`. When fewer than K documents were retrieved, `effective_k = min(k, document_count)`.

In [ ]:
results["relevance_at_5"].details

## 7. Hit Rate@K interpretation

Hit Rate@K is `1.0` when at least one relevant document occurs in the top effective K, otherwise `0.0`.

In [ ]:
results["hit_rate_at_5"].details

## 8. MRR@K interpretation

For one case, MRR@K is `1 / first_relevant_rank`, or `0.0` when none is relevant. Dataset/report aggregation may average these per-query scores later.

In [ ]:
results["mrr_at_5"].details

## 9. nDCG@K interpretation

nDCG@K discounts relevant documents by rank and compares the actual ordering with the ideal ordering of the same binary relevance values. Python calculates DCG, IDCG, and nDCG; all-irrelevant results receive `0.0`.

In [ ]:
results["ndcg_at_5"].details

## 10. One shared relevance call

For one case, the framework judges all required ranked documents in one structured LLM call through the deepest selected K. The judge returns only `rank`, binary `relevant`, and `reason`; Python derives all four metric scores. Four metrics at K=5 still make one relevance call, not twenty calls. Use `verbose=True` on an evaluator to include document text and reasons in that metric's diagnostics; scoring is unchanged.

## 11. Select only needed retrieval metrics

In [ ]:
selected = framework.evaluate(
    case, metrics=["relevance_at_5", "ndcg_at_5"]
)
selected["relevance_at_5"]
selected["ndcg_at_5"]

The selected metrics above still reuse one relevance call. Unselected metrics make no call. Unknown metric names raise before judge work.

## 12. Different K values

In [ ]:
mixed_k_framework = EvaluationFramework(
    judge=judge,
    evaluators=[
        RelevanceAtKEvaluator(k=3),
        HitRateAtKEvaluator(k=5),
        MRRAtKEvaluator(k=5),
        NDCGAtKEvaluator(k=10),
    ],
)
mixed_results = mixed_k_framework.evaluate(
    case, metrics=["hit_rate_at_5", "mrr_at_5"]
)

Only documents through the maximum selected K are judged. Here that is top 5; the configured but unselected nDCG@10 does not deepen the call.

## 13. Async

Jupyter supports top-level `await`. The one holistic relevance call consumes one slot from the framework's shared judge-call concurrency limit.

In [ ]:
async_results = await framework.a_evaluate(
    case, metrics=["relevance_at_5", "ndcg_at_5"], max_concurrency=4
)

## 14. `evaluate_many()`

Each case is a separate query and receives its own one-call relevance pass. Sync and async bulk methods preserve input order; async uses one shared concurrency limit across cases.

In [ ]:
second_case = EvaluationCase(
    case_id="retrieval-002",
    input="Where can I find billing statements?",
    retrieved_documents=documents,
)
cases = [case, second_case]
many_results = framework.evaluate_many(cases)
async_many_results = await framework.a_evaluate_many(
    cases, metrics=["relevance_at_5", "ndcg_at_5"], max_concurrency=4
)

## 15. Close resources

In [ ]:
judge.close()